# 14 · CP factorization and the rank-1 view / Factorización CP y la vista de rango 1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/14-cp-factorization.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#65a30d">DEEP DIVE · TAKE-HOME / ESTUDIO A FONDO · PARA DESPUÉS</span>

Section 11 fits a CP decomposition and compares its storage against Tucker's. This deep dive asks the other question:

> **What is a single CP component, and what can you read off it?**

The answer is the reason people reach for CP rather than Tucker. One component is one vector per axis, and each of those vectors is a profile over its own axis — where it peaks, where it dips, and what that means. A Tucker core mixes every basis vector with every other, so there is no component to read.

We will fit CP **by hand**, because the way it is fitted is the lesson. The problem is not convex in all the factors at once. Fix every factor but one and it becomes an ordinary least-squares problem — the one from section 07 — with a closed-form answer. That is **alternating least squares**, and it is the whole algorithm.

The data is Tamara Kolda's own demo tensor: 43 neurons × 200 time steps × 88 trials, recorded while a monkey moved a cursor to one of four targets.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 .7em">La sección 11 ajusta una descomposición CP y compara su almacenamiento con el de Tucker. Este estudio a fondo hace la otra pregunta:</div><div style="margin:0 0 .7em"><b>¿Qué es una sola componente CP y qué puedes leer en ella?</b></div><div style="margin:0 0 .7em">La respuesta es la razón por la que se elige CP antes que Tucker. Una componente es un vector por eje, y cada uno de esos vectores es un perfil sobre su propio eje: dónde sube, dónde baja y qué significa eso. El núcleo de Tucker mezcla cada vector base con todos los demás, así que no hay componente que leer.</div><div style="margin:0 0 .7em">Ajustaremos CP <b>a mano</b>, porque la forma de ajustarlo es la lección. El problema no es convexo en todos los factores a la vez. Fija todos los factores menos uno y se convierte en un problema de mínimos cuadrados corriente — el de la sección 07 — con solución cerrada. Eso es <b>mínimos cuadrados alternos</b>, y es todo el algoritmo.</div><div style="margin:0 0 0">Los datos son el tensor de demostración de la propia Tamara Kolda: 43 neuronas × 200 instantes × 88 ensayos, grabados mientras un mono movía un cursor hacia uno de cuatro objetivos.</div></div>

## What you will be able to do / Lo que podrás hacer

- Write a **rank-1 tensor** as an outer product `a ∘ b ∘ c` and say what each vector means.
- Read **peaks and troughs** in a factor vector as a profile over that vector's own axis.
- Explain why CP gives you components to read and **Tucker does not**.
- Fix two factors and recognise the third subproblem as the **least squares** of section 07.
- Implement **alternating least squares** yourself with a Khatri–Rao product and a pseudoinverse.
- Explain why **non-negativity** changes the subproblem to NNLS and keeps it convex.
- Fit non-negative CP to a real neural tensor and find a behavioural variable **the method never saw**.
- Tell **essential uniqueness** apart from what a run of ALS actually returns.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0">Escribir un <b>tensor de rango 1</b> como producto exterior <code>a ∘ b ∘ c</code> y decir qué significa cada vector.</li><li style="margin:.35em 0">Leer los <b>picos y valles</b> de un vector factor como un perfil sobre su propio eje.</li><li style="margin:.35em 0">Explicar por qué CP da componentes que se pueden leer y <b>Tucker no</b>.</li><li style="margin:.35em 0">Fijar dos factores y reconocer el tercer subproblema como los <b>mínimos cuadrados</b> de la sección 07.</li><li style="margin:.35em 0">Programar tú mismo <b>mínimos cuadrados alternos</b> con un producto Khatri–Rao y una pseudoinversa.</li><li style="margin:.35em 0">Explicar por qué la <b>no negatividad</b> convierte el subproblema en NNLS y lo mantiene convexo.</li><li style="margin:.35em 0">Ajustar CP no negativo a un tensor neuronal real y encontrar una variable de conducta <b>que el método nunca vio</b>.</li><li style="margin:.35em 0">Distinguir la <b>unicidad esencial</b> de lo que realmente devuelve una ejecución de ALS.</li></ul></div>

<!-- CORE-PATH -->
## Core path / Ruta esencial

Take-home: Exercise 1. Make the **Predict first** attempt below, run **Core prep**, then run the feedback helper before **Core activity**. Exercises 2–5 and the explorers are optional.

**You can:**

- Write a rank-1 tensor as `a ∘ b ∘ c` and say what each vector means.
- Explain why a factor vector's size means nothing until you normalise it.

<details>
<summary>Español · Ruta y metas</summary>

Para después del taller: Ejercicio 1. Responde primero a **Predice primero** abajo, ejecuta **Core prep** y después el comprobador antes de **Core activity**. Los ejercicios 2–5 y los exploradores son opcionales.

**Al terminar puedes:**

- Escribir un tensor de rango 1 como `a ∘ b ∘ c` y decir qué significa cada vector.
- Explicar por qué el tamaño de un vector factor no significa nada hasta normalizarlo.

</details>

## Predict first / Predice primero

**Retrieve:** In Notebook 11 you fitted a CP decomposition and read one column of a factor matrix. What did that column describe — a number, an axis, or a position along an axis?

Before reading the definitions or running the reveal, use these deliberately tiny synthetic values: `a = [1, 2]`, `b = [3, 4]`, `c = [5]`.

1. The rank-1 tensor is `T[i, j, k] = a[i] · b[j] · c[k]`. Write out all four numbers.
2. Now double `a` and halve `b`. Write out the four numbers again.
3. Did the tensor change? Did the vectors change? Which of the two is the thing you are allowed to interpret?

<details>
<summary>Español · Recupera y predice</summary>

**Recupera:** En el cuaderno 11 ajustaste una descomposición CP y leíste una columna de una matriz factor. ¿Qué describía esa columna: un número, un eje o una posición a lo largo de un eje?

Antes de leer las definiciones o ejecutar la respuesta, usa estos valores sintéticos deliberadamente pequeños: `a = [1, 2]`, `b = [3, 4]`, `c = [5]`.

1. El tensor de rango 1 es `T[i, j, k] = a[i] · b[j] · c[k]`. Escribe los cuatro números.
2. Ahora duplica `a` y divide `b` por dos. Escribe otra vez los cuatro números.
3. ¿Cambió el tensor? ¿Cambiaron los vectores? ¿Cuál de los dos es lo que tienes derecho a interpretar?

</details>

Four numbers / Cuatro números: ___ · After rescaling / Tras reescalar: ___

In [ ]:
#@title 🤔 Predict: does rescaling two factor vectors change the tensor? / Predice: ¿reescalar dos vectores factor cambia el tensor? — run me / ejecútame { display-mode: 'form' }

# --- counterexample / contraejemplo (tested in tests/test_teaching_materials.py) ---
import numpy as np

pred_a = np.array([1., 2.])
pred_b = np.array([3., 4.])
pred_c = np.array([5.])
pred_t1 = np.einsum("i,j,k->ijk", pred_a, pred_b, pred_c)
pred_t2 = np.einsum("i,j,k->ijk", 2 * pred_a, pred_b / 2, pred_c)

assert np.allclose(pred_t1, pred_t2)            # the tensor did not move
assert not np.allclose(2 * pred_a, pred_a)      # the vectors did

pred_na = np.linalg.norm(pred_a)
pred_nb = np.linalg.norm(pred_b)
pred_nc = np.linalg.norm(pred_c)
pred_lam = pred_na * pred_nb * pred_nc
pred_unit = np.einsum("i,j,k->ijk", pred_a / pred_na, pred_b / pred_nb,
                      pred_c / pred_nc)
assert np.allclose(pred_t1, pred_lam * pred_unit)   # all the size in one number
# --- end counterexample / fin del contraejemplo ---

import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import contextlib
import html as pred_html
import io

PRED_ACCENT = "#65a30d"
PRED_SANS = "ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"
PRED_MONO = "ui-monospace,SFMono-Regular,Menlo,Consolas,monospace"


def pred_tag(text):
    """A small EN / ES marker, in words rather than in colour alone."""
    return (f'<span style="font:700 10px/1 {PRED_MONO};letter-spacing:.16em;'
            f'color:{PRED_ACCENT};opacity:.8;margin-right:9px;'
            f'vertical-align:.12em">{text}</span>')


def pred_is_measurement(line):
    """True for a printed reading, false for a sentence."""
    if ":" not in line:
        return False
    tail = line.rsplit(":", 1)[1].strip()
    return bool(tail) and (tail[0].isdigit()
                           or tail[0] in "[(-+."
                           or tail.startswith(("True", "False", "nan", "inf")))


def pred_panel(text):
    """The reveal, laid out instead of printed. Same words, given typography."""
    blocks = []
    for line in text.rstrip("\n").split("\n"):
        stripped = line.strip()
        if not stripped:
            blocks.append('<div style="height:12px"></div>')
        elif stripped.startswith(("EN:", "ES:")):
            tag, body = stripped[:2], stripped[3:].strip()
            blocks.append(
                f'<p style="margin:.55em 0;font:400 15px/1.8 {PRED_SANS}">'
                f'{pred_tag(tag)}{pred_html.escape(body)}</p>')
        elif pred_is_measurement(stripped):
            blocks.append(
                f'<div style="font:400 13.5px/2.0 {PRED_MONO};'
                f'white-space:pre-wrap">{pred_html.escape(stripped)}</div>')
        else:
            blocks.append(
                f'<p style="margin:.55em 0;font:600 15.5px/1.75 {PRED_SANS}">'
                f'{pred_html.escape(stripped)}</p>')
    return (f'<div style="border-left:4px solid {PRED_ACCENT};'
            f'background:rgba(130,130,150,.08);border-radius:0 10px 10px 0;'
            f'padding:16px 20px;margin:.4em 0 0">{"".join(blocks)}</div>')


def pred_render(choice, reveal):
    """Run the check, catch what it prints, and show it laid out."""
    caught = io.StringIO()
    with contextlib.redirect_stdout(caught):
        check_prediction(choice, reveal)
    display(widgets.HTML(pred_panel(caught.getvalue())))


pred_choice = widgets.RadioButtons(
    options=[
        ("— choose one / elige una —", None),
        ("The tensor changes / El tensor cambia", "changes"),
        ("The tensor is unchanged / El tensor no cambia", "same"),
        ("Only its sign changes / Solo cambia su signo", "sign"),
    ],
    value=None,
    description="",
    layout=widgets.Layout(width="auto", margin="0 0 6px 0"),
)

pred_reveal = widgets.Checkbox(
    value=False,
    description="Show me the answer / Muéstrame la respuesta",
    indent=False,
    layout=widgets.Layout(margin="10px 0 4px 0"),
)


def check_prediction(choice, reveal):
    if choice is None:
        print("Choose an answer first / Elige una respuesta primero.")
        return

    if not reveal:
        print("Answer saved / Respuesta guardada.")
        print("Tick the box above when you are ready / Marca la casilla de "
              "arriba cuando quieras.")
        return

    print("a, b, c:", pred_a, pred_b, pred_c)
    print("2a, b/2, c:", 2 * pred_a, pred_b / 2, pred_c)
    print()
    print("tensor from a, b, c:", pred_t1.ravel())
    print("tensor from 2a, b/2, c:", pred_t2.ravel())
    print("same tensor? / ¿mismo tensor?", bool(np.allclose(pred_t1, pred_t2)))
    print()
    print("lam = |a| |b| |c|:", round(float(pred_lam), 4))
    print("lam * unit vectors:", (pred_lam * pred_unit).ravel())
    print()
    if choice == "same":
        print("You were right / Acertaste.")
    else:
        print("You were wrong — read on / Te equivocaste; sigue leyendo.")
    print()
    print("EN: a factor vector's magnitude is not a property of the data. Any "
          "scale you take out of one vector can be put back into another, and "
          "the rank-1 tensor is identical. So the only readable things are the "
          "normalised profile — the shape of the peaks and troughs — and one "
          "number, lam, holding all the size. Compare two components by lam "
          "and nothing else; compare two profiles only after normalising.")
    print("ES: la magnitud de un vector factor no es una propiedad de los "
          "datos. Cualquier escala que quites de un vector puedes devolverla a "
          "otro, y el tensor de rango 1 es idéntico. Así que lo único legible "
          "es el perfil normalizado — la forma de los picos y valles — y un "
          "número, lam, que guarda todo el tamaño. Compara dos componentes por "
          "lam y por nada más; compara dos perfiles solo tras normalizar.")


# The one <style> block in these notebooks, and the markdown rule does not
# cover it: this is *widget output*, not a markdown cell. Colab strips <style>
# from markdown -- which is why every box here is inline-styled -- but renders
# it in an output, the same path pandas' own Styler uses. Scoped to one added
# class, and if it is ever dropped the options still work.
pred_choice.add_class("pred-radio")

display(widgets.HTML(
    "<style>"
    ".pred-radio .widget-radio-box label{display:flex;align-items:flex-start;"
    "margin:0 0 13px;font:400 15px/1.6 " + PRED_SANS + "}"
    ".pred-radio input[type=radio]{flex:none;margin:4px 11px 0 0;"
    "transform:scale(1.15)}"
    "</style>"
))

pred_output = widgets.interactive_output(
    pred_render,
    {"choice": pred_choice, "reveal": pred_reveal},
)

pred_heading = widgets.HTML(
    f'<div style="font:700 11px/1.6 {PRED_MONO};letter-spacing:.18em;'
    f'color:{PRED_ACCENT};margin:2px 0 12px">'
    f'YOUR PREDICTION \u00b7 TU PREDICCI\u00d3N</div>'
)

display(widgets.VBox(
    [pred_heading, pred_choice, pred_reveal, pred_output],
    layout=widgets.Layout(padding="2px 0 14px 0"),
))

## Setup / Preparación

Three short cells: one install, one set of imports, one download. Together they take under a minute.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Tres celdas cortas: una instalación, un bloque de <code>import</code> y una descarga. Juntas tardan menos de un minuto.</div>

### Core prep 1/3 · Preparación esencial

Run the next cell. / Ejecuta la siguiente celda.

In [ ]:
%pip install -q tensorly

### Core prep 2/3 · Preparación esencial

Run the next cell. / Ejecuta la siguiente celda.

In [ ]:
import io
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import tensorly as tl
from scipy.io import loadmat
from scipy.optimize import nnls
from tensorly.decomposition import non_negative_parafac_hals

tl.set_backend("numpy")
rng = np.random.default_rng(14)

print("Imports / Importaciones: OK")

### Core prep 3/3 · Preparación esencial

Run the next cell. It downloads about 5 MB once.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Ejecuta la siguiente celda. Descarga unos 5 MB una sola vez.</div>

In [ ]:
# Tamara Kolda's own demo tensor. A monkey moved a cursor to one of four
# targets while 43 neurons were recorded; every trial is 200 time steps long.
# El tensor de demostración de la propia Tamara Kolda.
MONKEY_URL = "https://gitlab.com/tensors/tensor_data_monkey_bmi/-/raw/main/data.mat"


def fetch_monkey(url=MONKEY_URL, tries=3):
    """The .mat file as a dict. Retries, then fails in a sentence."""
    for attempt in range(1, tries + 1):
        try:
            with urllib.request.urlopen(url, timeout=120) as response:
                return loadmat(io.BytesIO(response.read()))
        except Exception as error:
            if attempt == tries:
                raise RuntimeError(
                    "EN: could not download the monkey BMI tensor. Check your "
                    "connection and run this cell again. / ES: no se pudo "
                    "descargar el tensor BMI del mono. Revisa tu conexión y "
                    "vuelve a ejecutar esta celda.") from error
            time.sleep(2 ** attempt)


monkey = fetch_monkey()
X = np.ascontiguousarray(monkey["X"])      # neurons x time x trials
angle = monkey["angle"].ravel()            # the target reached for, per trial

# A first, free rank-1 guess: the mean profile along each axis. It is not the
# best rank-1 fit -- Exercise 2 will beat it -- but it is a real one.
a = X.mean(axis=(1, 2))                    # (43,)  over neurons
b = X.mean(axis=(0, 2))                    # (200,) over time
c = X.mean(axis=(0, 1))                    # (88,)  over trials

print("Tensor / Tensor:", X.shape)
print("Targets / Objetivos:", np.unique(angle))
print("Factor vectors / Vectores factor:", a.shape, b.shape, c.shape)

## One component is one vector per axis / Una componente es un vector por eje

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

A CP decomposition writes a tensor as a sum of **rank-1 terms**. Each term is an outer product of one vector per axis, and a scalar holding its size:

$$
\mathcal{T} \;\approx\; \sum_{r=1}^{R} \lambda_r \; a_r \circ b_r \circ c_r
$$

Read one entry at a time and the outer product is just a product:

$$
\mathcal{T}[i, j, k] \;\approx\; \sum_{r=1}^{R} \lambda_r \; a_r[i] \, b_r[j] \, c_r[k]
$$

For our tensor, `a` has one number per **neuron**, `b` has one per **time step**, and `c` has one per **trial**. So a single component says: *these neurons* (wherever `a` peaks), *at this point in the trial* (wherever `b` peaks), *on these trials* (wherever `c` peaks). Three profiles, one story, and every one of them is a vector you can plot and point at.

That is what CP gives you and Tucker does not. A Tucker core mixes every basis vector of one mode with every basis vector of the others, so there is no single term to lift out and read. The price CP pays is that it cannot be computed by a sequence of SVDs — which is the next section.

The factor matrices collect the vectors column by column: `A` is `(43, R)`, `B` is `(200, R)`, `C` is `(88, R)`. Column `r` of each one belongs to component `r`.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una descomposición CP escribe el tensor como suma de <b>términos de rango 1</b>. Cada término es un producto exterior de un vector por eje, más un escalar con su tamaño. En nuestro tensor, <code>a</code> tiene un número por <b>neurona</b>, <code>b</code> uno por <b>instante</b> y <code>c</code> uno por <b>ensayo</b>. Una sola componente dice: <i>estas neuronas</i>, <i>en este momento del ensayo</i>, <i>en estos ensayos</i>. Tres perfiles, una historia, y cada uno es un vector que puedes dibujar y señalar.
<br><br>
Eso es lo que da CP y no da Tucker: el núcleo de Tucker mezcla cada vector base de un modo con los de los demás, así que no hay un término suelto que leer. El precio que paga CP es que no se calcula con una secuencia de SVD, que es la sección siguiente.
<br><br>
Las matrices factor reúnen los vectores por columnas: <code>A</code> es <code>(43, R)</code>, <code>B</code> es <code>(200, R)</code>, <code>C</code> es <code>(88, R)</code>. La columna <code>r</code> de cada una pertenece a la componente <code>r</code>.</div>

### Three peaks, and where they meet / Tres picos, y dónde se encuentran

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

The first three frames are the component's three vectors, each with its own peak lit: <code>a</code> peaks at neuron 0, <code>b</code> at time step 2, <code>c</code> at trial 3. The last frame is the tensor they build, with the one cell those three peaks meet at — <code>4 · 5 · 4 = 80</code>. That cell is the component's loudest moment, and its address is read straight off the three argmaxes.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-14-profile.gif" alt="An animation in four frames. The first three each show one short vector of a CP component — over neurons, over time, over trials — with its largest entry highlighted and its index given. The fourth frame shows the rank-1 tensor the three build, with a single cell highlighted: the entry whose three indices are the three highlighted positions, labelled T[0, 2, 3] = 80." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Los tres primeros fotogramas son los tres vectores de la componente, cada uno con su propio pico iluminado: <code>a</code> culmina en la neurona 0, <code>b</code> en el instante 2, <code>c</code> en el ensayo 3. El último fotograma es el tensor que construyen, con la única celda donde se encuentran esos tres picos: <code>4 · 5 · 4 = 80</code>. Esa celda es el momento más alto de la componente, y su dirección se lee directamente de los tres argmax.</div>

### Fix two, solve one / Fija dos, resuelve uno

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

Alternating least squares in four frames. Each frame holds two factor matrices fixed — drawn pale — and solves for the third, which lights up. The fixed pair turns the problem into an ordinary least-squares fit, the one from section 07. Then the sweep moves on.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-14-als.gif" alt="An animation of the three factor matrices A, B and C. In each of the first three frames two of them are pale and one is highlighted, captioned with the closed-form least-squares update being solved for the highlighted one. In the fourth frame all three are highlighted together, captioned one sweep, three least squares." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Mínimos cuadrados alternos en cuatro fotogramas. Cada fotograma mantiene fijas dos matrices factor — dibujadas en pálido — y resuelve la tercera, que se ilumina. El par fijo convierte el problema en un ajuste de mínimos cuadrados corriente, el de la sección 07. Después el barrido continúa.</div>

### Essentially unique, and what that gives away / Esencialmente única, y qué concede

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

Run ALS twice and you get two different-looking factor sets, which makes \"unique\" sound like a lie. It is not. Exactly two things are free — which component is called first, and how a component's size is split between its three vectors — and both leave the tensor identical. The directions are what is pinned down, and the directions are the part worth having.

$$
\sum_{r} a_r \circ b_r \circ c_r
= \sum_{r} (\alpha_r a_r) \circ (\beta_r b_r) \circ (\gamma_r c_r)
\qquad\text{whenever}\qquad
\alpha_r \beta_r \gamma_r = 1
$$

Read it as: you may rescale the three vectors of a component however you like
so long as the three factors multiply to one, and you may list the components
in any order. Nothing else is free — which is why a CP component can be named,
and an NMF factor, which can also be rotated, cannot.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-14-unique.gif" alt="An animation of three 2 by 2 factor matrices A, B and C, and the 2 by 2 by 2 tensor their two components build. The columns of all three are then swapped and the tensor is unchanged. Finally the first column of A is doubled and the first column of B halved, and the tensor is unchanged again." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:11px">🇪🇸 ESPAÑOL</div>Ejecuta ALS dos veces y obtienes dos conjuntos de factores de aspecto distinto, lo que hace que «única» suene a mentira. No lo es. Exactamente dos cosas son libres —qué componente se nombra primero, y cómo se reparte el tamaño de una componente entre sus tres vectores— y ambas dejan el tensor idéntico. Las direcciones son lo que queda fijado, y las direcciones son la parte que vale la pena.</div>

In [ ]:
#@title ⏸️ Step through the animations / Recorre las animaciones { display-mode: 'form' }

# Plumbing, not a lesson. The two animations above loop forever and a GIF
# cannot be paused -- so this fetches the same frames and hands them over one
# at a time, at whatever pace you read at.
# Plomería, no una lección: trae los mismos fotogramas y los entrega de uno en
# uno, al ritmo al que leas.

import io
import urllib.request

import ipywidgets as widgets
from IPython.display import display
from PIL import Image

gif_urls = [
    "https://project-delphi.github.io/tensors-workshop/images/cube-14-profile.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-14-als.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-14-unique.gif",
]


def gif_frames(url):
    """Every frame of an animated GIF, as PNG bytes."""
    with urllib.request.urlopen(url, timeout=30) as response:
        gif = Image.open(io.BytesIO(response.read()))
    out = []
    try:
        while True:
            buffer = io.BytesIO()
            gif.convert("RGB").save(buffer, format="PNG")
            out.append(buffer.getvalue())
            gif.seek(gif.tell() + 1)
    except EOFError:
        pass
    return out


try:
    gif_cache = {url: gif_frames(url) for url in gif_urls}
except Exception as error:  # offline, or the site is down
    print("EN: could not reach the site, so there are no frames to step "
          "through.", error)
    print("ES: no se pudo acceder al sitio, así que no hay fotogramas que "
          "recorrer.", error)
else:
    gif_pick = widgets.Dropdown(
        options=[(url.rsplit("/", 1)[1], url) for url in gif_urls],
        description="Animation / Animación:",
        style={"description_width": "180px"},
    )
    gif_step = widgets.IntSlider(
        min=1, max=len(gif_cache[gif_urls[0]]), value=1,
        description="Frame / Fotograma:",
        style={"description_width": "180px"},
        continuous_update=False,
    )
    gif_prev = widgets.Button(description="◀ Prev")
    gif_next = widgets.Button(description="Next ▶")
    # An Image widget, deliberately, and never widgets.Output: a payload
    # leaving an Output widget makes nbclient wait out the whole cell timeout
    # (see scripts/test_notebooks.py). This one is a plain bytes trait.
    gif_view = widgets.Image(format="png",
                             layout=widgets.Layout(max_width="100%"))

    def gif_show(*_):
        frames = gif_cache[gif_pick.value]
        gif_step.max = len(frames)
        gif_view.value = frames[min(gif_step.value, len(frames)) - 1]

    def gif_bump(delta):
        def click(_):
            frames = gif_cache[gif_pick.value]
            gif_step.value = (gif_step.value - 1 + delta) % len(frames) + 1
        return click

    gif_prev.on_click(gif_bump(-1))
    gif_next.on_click(gif_bump(+1))
    gif_pick.observe(gif_show, names="value")
    gif_step.observe(gif_show, names="value")
    gif_show()

    display(widgets.VBox([
        gif_pick,
        widgets.HBox([gif_prev, gif_step, gif_next]),
        gif_view,
    ]))

## Exercise 1 — build a rank-1 tensor and read it / Ejercicio 1 — construye un tensor de rango 1 y léelo

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

`a`, `b` and `c` are already in memory: the mean profile of the data along each axis. They are a real rank-1 component, just not the best one. This exercise is about what you are allowed to say about them.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>a</code>, <code>b</code> y <code>c</code> ya están en memoria: el perfil medio de los datos a lo largo de cada eje. Son una componente de rango 1 real, aunque no la mejor. Este ejercicio trata de qué tienes derecho a decir sobre ellos.</div>

In [ ]:
# Feedback helper / Comprobador: run me before the Core activity.
# Ejecútame antes de la actividad esencial.


def check_rank_one(R1, R2, an, bn, cn, lam, peak_t):
    """Says what is right, what is wrong, and what to look at next."""
    notes = []

    def ok(condition, good, bad):
        notes.append(("✅ " + good) if condition else ("❌ " + bad))

    ok(np.shape(R1) == X.shape,
       f"R1 has the right shape {X.shape}.",
       f"R1 is {np.shape(R1)}; it should be {X.shape} — one axis per vector.")
    ok(np.allclose(R1, R2),
       "R1 and R2 are the same tensor: the rescaling cancelled.",
       "R1 and R2 differ. Doubling one vector and halving another should "
       "leave the product untouched — check which vectors you rescaled.")
    for name, vec, want in (("an", an, len(a)), ("bn", bn, len(b)),
                            ("cn", cn, len(c))):
        ok(np.shape(vec) == (want,) and abs(np.linalg.norm(vec) - 1) < 1e-8,
           f"{name} is a unit vector of length {want}.",
           f"{name} should have length {want} and norm 1; it has shape "
           f"{np.shape(vec)} and norm {np.linalg.norm(vec):.4g}.")
    ok(np.allclose(R1, lam * np.einsum("i,j,k->ijk", an, bn, cn)),
       f"lam = {float(lam):.4g} puts all the size back: "
       "lam · (an ∘ bn ∘ cn) rebuilds R1.",
       "lam · (an ∘ bn ∘ cn) does not rebuild R1. lam is the product of the "
       "three norms you divided out.")
    ok(int(peak_t) == int(np.argmax(bn)),
       f"peak_t = {int(peak_t)} is where the time profile peaks.",
       f"peak_t should be argmax of the time profile, which is "
       f"{int(np.argmax(bn))}.")

    print("\n".join(notes))
    print()
    print("EN: normalising is not tidying up. Until you do it, 'this vector is "
          "bigger' is a statement about your arithmetic, not about the monkey.")
    print("ES: normalizar no es ordenar. Hasta que lo haces, «este vector es "
          "mayor» habla de tu aritmética, no del mono.")

### Core activity · Actividad esencial

**Predict → Run → Explain → Check**

1. **Predict.** Write down, before running anything, whether `2a ∘ (b/2) ∘ c` is the same tensor as `a ∘ b ∘ c`, and what `R1.shape` will be.
2. **Run.** Complete Exercise 1 below. Build `R1` and `R2` with `np.einsum`, normalise the three vectors into `an`, `bn`, `cn`, and put all the size into `lam`.
3. **Explain.** In one sentence, say what `lam` measures and what it would mean for one component to have a larger `lam` than another.
4. **Check.** Call `check_rank_one(R1, R2, an, bn, cn, lam, peak_t)`. Compare the result against your written prediction before opening the solution.

<details>
<summary>Español · Predice → Ejecuta → Explica → Comprueba</summary>

1. **Predice.** Anota, antes de ejecutar nada, si `2a ∘ (b/2) ∘ c` es el mismo tensor que `a ∘ b ∘ c`, y cuál será `R1.shape`.
2. **Ejecuta.** Completa el Ejercicio 1. Construye `R1` y `R2` con `np.einsum`, normaliza los tres vectores en `an`, `bn`, `cn` y pon todo el tamaño en `lam`.
3. **Explica.** En una frase, di qué mide `lam` y qué significaría que una componente tuviera un `lam` mayor que otra.
4. **Comprueba.** Llama a `check_rank_one(R1, R2, an, bn, cn, lam, peak_t)`. Compara con tu predicción escrita antes de abrir la solución.

</details>

Prediction / Predicción: ___  
Evidence / Evidencia: ___  
Revised explanation / Explicación revisada: ___

<details>
<summary>Hint 1 / Pista 1 · if stuck after an attempt / si te atascas tras intentarlo</summary>

`np.einsum("i,j,k->ijk", a, b, c)` builds the outer product directly. The output has one axis per input vector, in the order you name them.

`np.einsum("i,j,k->ijk", a, b, c)` construye el producto exterior directamente. La salida tiene un eje por vector de entrada, en el orden en que los nombres.

</details>

<details>
<summary>Hint 2 / Pista 2 · implementation / implementación</summary>

To normalise, divide each vector by `np.linalg.norm(...)`. The size you removed is the product of the three norms — that product is `lam`, and multiplying the unit outer product by it gets you back exactly where you started.

Para normalizar, divide cada vector por `np.linalg.norm(...)`. El tamaño que quitaste es el producto de las tres normas: ese producto es `lam`, y multiplicar por él el producto exterior unitario te devuelve exactamente al punto de partida.

</details>

In [ ]:
# TODO 1 / TAREA 1
#
# a (43,) over neurons, b (200,) over time, c (88,) over trials.
#
# 1. R1 = the rank-1 tensor a ∘ b ∘ c, built with np.einsum.
# 2. R2 = the same thing from 2*a, b/2 and c.
# 3. an, bn, cn = the three vectors normalised to unit length.
# 4. lam = the one number that puts the size back, so that
#    lam * (an ∘ bn ∘ cn) equals R1.
# 5. peak_t = the time step where the normalised time profile peaks.
#
# Then run: check_rank_one(R1, R2, an, bn, cn, lam, peak_t)

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }

R1 = np.einsum("i,j,k->ijk", a, b, c)
R2 = np.einsum("i,j,k->ijk", 2 * a, b / 2, c)

an = a / np.linalg.norm(a)
bn = b / np.linalg.norm(b)
cn = c / np.linalg.norm(c)
lam = np.linalg.norm(a) * np.linalg.norm(b) * np.linalg.norm(c)

peak_t = int(np.argmax(bn))

check_rank_one(R1, R2, an, bn, cn, lam, peak_t)

In [ ]:
# Visible on purpose. Everything below uses these names, and a reader who
# never opens the solution above still has to be able to run the rest.
# Visible a propósito: lo de abajo usa estos nombres.

an = a / np.linalg.norm(a)
bn = b / np.linalg.norm(b)
cn = c / np.linalg.norm(c)
lam = np.linalg.norm(a) * np.linalg.norm(b) * np.linalg.norm(c)
R1 = lam * np.einsum("i,j,k->ijk", an, bn, cn)

unit = np.einsum("i,j,k->ijk", an, bn, cn)
best = float(np.einsum("ijk,ijk->", X, unit))   # the scale that fits X best

print("lam / lam:", round(float(lam), 4))
print("time profile peaks at / el perfil temporal culmina en t =",
      int(np.argmax(bn)))
print("relative error as built / error relativo tal cual:",
      round(float(np.linalg.norm(X - R1) / np.linalg.norm(X)), 4))
print("best scale for these directions / mejor escala:", round(best, 4))
print("relative error at that scale / error con esa escala:",
      round(float(np.linalg.norm(X - best * unit) / np.linalg.norm(X)), 4))

<details>
<summary><strong>What did Exercise 1 show? / ¿Qué mostró el Ejercicio 1?</strong></summary>

Three things, in order of how often they are got wrong.

**The individual vectors have no scale of their own.** Any factor of 2 you take out of `a` can be put back into `b`, and the tensor does not move. So "neuron 12 has a big weight" is not a statement about the data until you have normalised. This is the same ambiguity CP always has, and libraries handle it by returning unit-norm factors plus a weight vector — `tensorly` calls it `weights`, and it is exactly our `lam`.

**After normalising, the profile is the readable object.** `bn` is a shape over time: where it rises, where it falls. Its peak is a moment in the trial. That is what a component *says*.

**Direction and size are two separate questions, and the cell above prints both answers.** As built, `a ∘ b ∘ c` has a relative error of about 1.00 — the mean of a product is not the product of the means, so the tensor comes out far too small. Rescale the same three directions to the best possible size and the error drops to about 0.49. Nothing about the *shape* of those profiles changed; only one number did. The 0.49 is the honest baseline, and beating it is the next exercise.

<br>

**ES.** Tres cosas, por orden de frecuencia del error. **Los vectores no tienen escala propia**: cualquier factor 2 que quites de `a` puedes devolverlo a `b` y el tensor no se mueve, así que «la neurona 12 pesa mucho» no dice nada hasta normalizar. Las librerías resuelven esto devolviendo factores de norma 1 más un vector de pesos: `tensorly` lo llama `weights` y es exactamente nuestro `lam`. **Tras normalizar, el perfil es lo legible**: `bn` es una forma en el tiempo, y su pico es un momento del ensayo. **Dirección y tamaño son dos preguntas distintas, y la celda de arriba imprime ambas respuestas**: tal cual, el error relativo ronda 1,00, porque la media de un producto no es el producto de las medias; reescalado al mejor tamaño posible, baja a unos 0,49. La *forma* de los perfiles no cambió, solo un número. Ese 0,49 es la referencia honesta, y batirlo es el ejercicio siguiente.

</details>

## The subproblem you already know / El subproblema que ya conoces

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

Fitting CP means choosing `A`, `B` and `C` to minimise

$$
\left\lVert \mathcal{T} - \sum_{r=1}^{R} a_r \circ b_r \circ c_r \right\rVert_F^2
$$

and that problem is **not convex** in all three factors at once — the model multiplies unknowns together, which is exactly what convexity forbids. There is no SVD-like recipe waiting for you, and unlike Tucker you cannot get there by decomposing one mode at a time.

Now fix `B` and `C` and look again. With the other two held still, every entry of the model is **linear** in `A`. Unfolding the tensor along mode 0 turns the whole thing into one matrix equation:

$$
\mathcal{T}_{(0)} \;\approx\; A \, (B \odot C)^{\mathsf{T}}
$$

where $\odot$ is the **Khatri–Rao** product: column $r$ of $B \odot C$ is the outer product of column $r$ of $B$ with column $r$ of $C$, flattened. That is an ordinary linear least-squares problem in `A`, it is convex, and section 07 already told you how to solve it — with a pseudoinverse:

$$
A \;=\; \mathcal{T}_{(0)} \, (B \odot C) \,\bigl[(B^{\mathsf{T}}B) * (C^{\mathsf{T}}C)\bigr]^{+}
$$

The $*$ there is elementwise multiplication, and it is a shortcut worth knowing: it computes $(B \odot C)^{\mathsf{T}}(B \odot C)$ as an $R \times R$ matrix without ever forming the tall Khatri–Rao product's Gram matrix.

**Alternating least squares** is that, three times, over and over: solve for `A` with `B` and `C` fixed, then `B`, then `C`, then start again. Every step is convex and every step can only lower the error. The sequence of steps is not guaranteed to reach the global minimum — that is the honest cost of the non-convexity — but it is guaranteed never to make things worse.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Ajustar CP es elegir <code>A</code>, <code>B</code> y <code>C</code> que minimicen el error, y ese problema <b>no es convexo</b> en los tres factores a la vez: el modelo multiplica incógnitas entre sí, que es justo lo que la convexidad prohíbe. No hay receta tipo SVD esperándote, y a diferencia de Tucker no se llega descomponiendo un modo cada vez.
<br><br>
Ahora fija <code>B</code> y <code>C</code>. Con los otros dos quietos, cada entrada del modelo es <b>lineal</b> en <code>A</code>, y desplegar el tensor por el modo 0 lo convierte en una sola ecuación matricial con el producto <b>Khatri–Rao</b>. Es un problema de mínimos cuadrados corriente, es convexo, y la sección 07 ya te dijo cómo resolverlo: con una pseudoinversa.
<br><br>
<b>Mínimos cuadrados alternos</b> es eso, tres veces, una y otra vez. Cada paso es convexo y cada paso solo puede bajar el error. La sucesión de pasos no garantiza llegar al mínimo global — ese es el coste honesto de la no convexidad — pero garantiza no empeorar nunca.</div>

## Exercise 2 — alternating least squares by hand / Ejercicio 2 — mínimos cuadrados alternos a mano

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

Optional, and the most useful cell in the notebook. Forty lines of NumPy, no library.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Opcional, y la celda más útil del cuaderno. Cuarenta líneas de NumPy, sin librería.</div>

In [ ]:
# TODO 2 / TAREA 2
#
# 1. unfold(T, mode) -> a matrix with T.shape[mode] rows.
#    Use np.moveaxis to bring `mode` to the front, then reshape.
#
# 2. khatri_rao(P, Q) -> the column-wise Kronecker product.
#    P is (I, R), Q is (J, R); the result is (I*J, R), and column r is the
#    outer product of P[:, r] and Q[:, r], flattened.
#
# 3. als_fit(T, rank, sweeps, seed) -> (A, B, C, errors)
#    Start from random non-negative factors. In each sweep, update A, then B,
#    then C, each with the closed form:
#        F = unfold(T, mode) @ M @ np.linalg.pinv((P.T @ P) * (Q.T @ Q))
#    where M = khatri_rao(P, Q) for the other two factors P, Q, in axis order.
#    Record the relative error after every sweep.
#
# 4. Fit rank 3 to X with 30 sweeps. Store A_als, B_als, C_als, err_als.
#
# Is err_als monotone? Does it beat the mean-profile error from Exercise 1?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }


def unfold(T, mode):
    """The mode-n unfolding: that axis becomes the rows, the rest the columns."""
    return np.moveaxis(T, mode, 0).reshape(T.shape[mode], -1)


def khatri_rao(P, Q):
    """Column-wise Kronecker product: (I, R) and (J, R) -> (I*J, R)."""
    return (P[:, None, :] * Q[None, :, :]).reshape(-1, P.shape[1])


def als_fit(T, rank, sweeps=30, seed=0):
    """CP by alternating least squares. Returns the factors and the error curve."""
    start = np.random.default_rng(seed)
    factors = [np.abs(start.standard_normal((dim, rank))) for dim in T.shape]
    norm_T = np.linalg.norm(T)
    errors = []

    for _ in range(sweeps):
        for mode in range(T.ndim):
            others = [factors[m] for m in range(T.ndim) if m != mode]
            M = khatri_rao(*others)
            gram = others[0].T @ others[0]
            for other in others[1:]:
                gram = gram * (other.T @ other)
            factors[mode] = unfold(T, mode) @ M @ np.linalg.pinv(gram)
        model = np.einsum("ir,jr,kr->ijk", *factors)
        errors.append(float(np.linalg.norm(T - model) / norm_T))

    return (*factors, np.array(errors))


A_als, B_als, C_als, err_als = als_fit(X, rank=3, sweeps=30, seed=0)

print("first sweep / primer barrido:", round(err_als[0], 4))
print("last sweep  / último barrido:", round(err_als[-1], 4))
print("monotone? / ¿monótona?", bool(np.all(np.diff(err_als) <= 1e-12)))

In [ ]:
# Visible on purpose: the explorers below need these names, and a reader who
# never opened the solution still has to be able to run them.
# Visible a propósito: los exploradores de abajo necesitan estos nombres.


def unfold(T, mode):
    """The mode-n unfolding: that axis becomes the rows, the rest the columns."""
    return np.moveaxis(T, mode, 0).reshape(T.shape[mode], -1)


def khatri_rao(P, Q):
    """Column-wise Kronecker product: (I, R) and (J, R) -> (I*J, R)."""
    return (P[:, None, :] * Q[None, :, :]).reshape(-1, P.shape[1])


def als_fit(T, rank, sweeps=30, seed=0):
    """CP by alternating least squares. Returns the factors and the error curve."""
    start = np.random.default_rng(seed)
    factors = [np.abs(start.standard_normal((dim, rank))) for dim in T.shape]
    norm_T = np.linalg.norm(T)
    errors = []
    for _ in range(sweeps):
        for mode in range(T.ndim):
            others = [factors[m] for m in range(T.ndim) if m != mode]
            M = khatri_rao(*others)
            gram = others[0].T @ others[0]
            for other in others[1:]:
                gram = gram * (other.T @ other)
            factors[mode] = unfold(T, mode) @ M @ np.linalg.pinv(gram)
        model = np.einsum("ir,jr,kr->ijk", *factors)
        errors.append(float(np.linalg.norm(T - model) / norm_T))
    return (*factors, np.array(errors))


A_als, B_als, C_als, err_als = als_fit(X, rank=3, sweeps=30, seed=0)
print("rank 3, 30 sweeps / rango 3, 30 barridos:",
      round(err_als[0], 4), "->", round(err_als[-1], 4))

### Interactive ALS explorer / Explorador interactivo de ALS

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

Rank and starting seed, against the error curve. Two things to look for: whether the curve is ever **not** monotone, and whether two seeds at the same rank land on the same error.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Rango y semilla inicial, frente a la curva de error. Dos cosas que mirar: si la curva deja de ser <b>monótona</b> alguna vez, y si dos semillas con el mismo rango llegan al mismo error.</div>

In [ ]:
#@title 📉 ALS explorer / Explorador de ALS { display-mode: 'form' }

import ipywidgets as widgets
from IPython.display import display

# Precomputed once: four ranks x three seeds, so moving a slider is a lookup
# rather than a fit. Precalculado una vez: mover un control es una consulta.
als_grid = {(r, s): als_fit(X, rank=r, sweeps=25, seed=s)[-1]
            for r in (1, 2, 3, 5) for s in (0, 1, 2)}


def als_draw(rank, seed, compare):
    curve = als_grid[(rank, seed)]
    figure, axes = plt.subplots(figsize=(7.2, 3.4))
    axes.plot(np.arange(1, len(curve) + 1), curve, lw=2.2, color="#65a30d",
              label=f"rank {rank}, seed {seed}")
    if compare:
        for other in (0, 1, 2):
            if other != seed:
                axes.plot(np.arange(1, len(curve) + 1), als_grid[(rank, other)],
                          lw=1.2, alpha=.55, color="#9ca3af",
                          label=f"rank {rank}, seed {other}")
    axes.set_xlabel("sweep / barrido")
    axes.set_ylabel("relative error / error relativo")
    axes.set_title(f"final / final: {curve[-1]:.4f}   "
                   f"monotone / monótona: {bool(np.all(np.diff(curve) <= 1e-12))}")
    axes.legend(fontsize=8)
    axes.grid(alpha=.25)
    figure.tight_layout()
    plt.show()


als_rank = widgets.SelectionSlider(options=[1, 2, 3, 5], value=3,
                                   description="rank / rango:",
                                   style={"description_width": "150px"})
als_seed = widgets.IntSlider(min=0, max=2, value=0, description="seed / semilla:",
                             style={"description_width": "150px"},
                             continuous_update=False)
als_compare = widgets.Checkbox(value=False, indent=False,
                               description="show the other seeds / muestra las "
                                           "otras semillas")

display(widgets.VBox([
    als_rank, als_seed, als_compare,
    widgets.interactive_output(
        als_draw,
        {"rank": als_rank, "seed": als_seed, "compare": als_compare}),
]))

<details>
<summary><strong>What did Exercise 2 show? / ¿Qué mostró el Ejercicio 2?</strong></summary>

**The error never goes up.** Each of the three updates in a sweep is the exact minimiser of a convex problem with the other two factors held still, so it cannot make the objective worse. That is not a property of the code; it is the reason the algorithm is written this way.

**It is not the same as finding the best answer.** Monotone descent guarantees you reach *a* stationary point. Change the seed in the explorer and, at higher ranks, you will find fits that stop at different errors. CP's non-convexity did not go away when we alternated — it moved into the choice of starting point.

**The pseudoinverse is doing the work you met in section 07.** `np.linalg.pinv` on an `R × R` matrix, three times per sweep. The Khatri–Rao product is what turns a tensor equation into the matrix equation that pseudoinverse can solve, and the Hadamard-product shortcut `(BᵀB) * (CᵀC)` is why the cost does not blow up: you never form the tall `(J·K, R)` matrix's Gram matrix directly.

<br>

**ES.** **El error nunca sube**: cada una de las tres actualizaciones de un barrido es el mínimo exacto de un problema convexo con los otros dos factores quietos, así que no puede empeorar el objetivo. No es una propiedad del código: es la razón de escribir así el algoritmo. **No es lo mismo que encontrar la mejor respuesta**: el descenso monótono garantiza llegar a *un* punto estacionario; cambia la semilla en el explorador y con rangos altos verás ajustes que se detienen en errores distintos. La no convexidad no desapareció al alternar, se mudó a la elección del punto de partida. **La pseudoinversa hace el trabajo de la sección 07**, sobre una matriz `R × R`, tres veces por barrido; el producto Khatri–Rao es lo que convierte la ecuación tensorial en la ecuación matricial que esa pseudoinversa resuelve.

</details>

## Non-negativity keeps the subproblem convex / La no negatividad conserva la convexidad

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

Look at the factors Exercise 2 returned and you will find negative numbers in all three. A firing rate cannot be negative, and neither can "how much of this pattern was present on trial 7". A component with a negative entry is still arithmetic that reconstructs the data; it is just no longer a **profile** you can read, because there is no such thing as a neuron contributing minus a spike.

Adding the constraint changes the subproblem, but not as much as you might fear. With `B` and `C` fixed, solving

$$
\min_{A \,\geq\, 0} \; \left\lVert \mathcal{T}_{(0)} - A \, (B \odot C)^{\mathsf{T}} \right\rVert_F^2
$$

is **non-negative least squares** — still a convex problem, one row of `A` at a time, and `scipy.optimize.nnls` solves it exactly. What is lost is the closed form: there is no pseudoinverse that respects an inequality, so each row is an iterative solve of its own. The alternating structure is untouched.

In practice you use a library, and `tensorly` offers two. Multiplicative updates are the classic; **HALS** — hierarchical alternating least squares — updates one *column* at a time and converges faster. Both are still "fix the rest, solve one piece, repeat".

Expect the constrained fit to be slightly **worse** in squared error than the unconstrained one. It has to be: the non-negative orthant is a subset of the space the unconstrained fit searched. What you buy with that error is a factor you can point at and describe.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Mira los factores del Ejercicio 2 y encontrarás números negativos en los tres. Una tasa de disparo no puede ser negativa, ni tampoco «cuánto de este patrón hubo en el ensayo 7». Una componente con una entrada negativa sigue siendo aritmética que reconstruye los datos; solo deja de ser un <b>perfil</b> legible, porque no existe una neurona que aporte menos un disparo.
<br><br>
Añadir la restricción cambia el subproblema menos de lo que parece: con <code>B</code> y <code>C</code> fijos, el problema es de <b>mínimos cuadrados no negativos</b>, sigue siendo convexo, y <code>scipy.optimize.nnls</code> lo resuelve exactamente fila a fila. Lo que se pierde es la forma cerrada: ninguna pseudoinversa respeta una desigualdad. La estructura alterna queda intacta.
<br><br>
En la práctica se usa una librería. <code>tensorly</code> ofrece dos: las actualizaciones multiplicativas, clásicas, y <b>HALS</b>, que actualiza una <i>columna</i> cada vez y converge más rápido. Espera que el ajuste restringido sea algo <b>peor</b> en error cuadrático: tiene que serlo, porque el ortante no negativo es un subconjunto del espacio que buscaba el ajuste libre. Lo que compras con ese error es un factor que puedes señalar y describir.</div>

## Exercise 3 — non-negative CP / Ejercicio 3 — CP no negativo

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Opcional. Primero mira lo que hay que arreglar, después arréglalo dos veces: a mano en un modo, y con la librería en los tres.</div>

In [ ]:
# TODO 3 / TAREA 3
#
# 1. neg_counts = how many negative entries each of A_als, B_als, C_als holds.
#    Store a list of three integers.
#
# 2. nnls_update(T, mode, factors) -> the non-negative mode update, one row at
#    a time with scipy.optimize.nnls. `factors` is all three in axis order;
#    build the Khatri-Rao product from the two that are not `mode`, in that
#    same order, or the columns will not line up with unfold(T, mode).
#
# 3. Apply it to mode 0, with B_als and C_als fixed. Call the result A_pos.
#    Confirm it has no negative entries, and compare its residual against the
#    unconstrained A_als on the same fixed B and C.
#
# 4. Fit the real thing at rank 3 with non_negative_parafac_hals:
#       weights_nn, factors_nn = non_negative_parafac_hals(
#           tl.tensor(X), rank=3, init="random", random_state=0,
#           n_iter_max=300, tol=1e-8, normalize_factors=True)
#    Unpack factors_nn into A_nn, B_nn, C_nn and store the relative error in
#    err_nn. Is it better or worse than err_als[-1]? Should it be?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }

neg_counts = [int((F < 0).sum()) for F in (A_als, B_als, C_als)]
print("negative entries / entradas negativas:", neg_counts)


def nnls_update(T, mode, factors):
    """One non-negative mode update: NNLS on each row, exactly.

    `factors` is all three, in axis order; the two that are not `mode` are what
    the Khatri-Rao product is built from, and their order has to match the
    column order of `unfold(T, mode)`. Taking them from the list rather than as
    two arguments is what keeps those two in step -- passing P and Q by hand
    works for mode 0 and silently returns the wrong factor for 1 and 2.
    """
    others = [factors[m] for m in range(T.ndim) if m != mode]
    M = khatri_rao(*others)
    return np.stack([nnls(M, row)[0] for row in unfold(T, mode)])


A_pos = nnls_update(X, 0, [A_als, B_als, C_als])
M0 = khatri_rao(B_als, C_als)
X0 = unfold(X, 0)
print("A_pos has negatives? / ¿tiene negativos?", bool((A_pos < 0).any()))
print("residual, constrained / residuo, restringido:",
      round(float(np.linalg.norm(X0 - A_pos @ M0.T)), 4))
print("residual, unconstrained / residuo, libre:",
      round(float(np.linalg.norm(X0 - A_als @ M0.T)), 4))

weights_nn, factors_nn = non_negative_parafac_hals(
    tl.tensor(X), rank=3, init="random", random_state=0,
    n_iter_max=300, tol=1e-8, normalize_factors=True)
A_nn, B_nn, C_nn = factors_nn
err_nn = float(np.linalg.norm(X - tl.cp_to_tensor((weights_nn, factors_nn)))
               / np.linalg.norm(X))

print()
print("non-negative CP error / error de CP no negativo:", round(err_nn, 4))
print("unconstrained ALS error / error de ALS libre:", round(err_als[-1], 4))
print("weights / pesos:", np.round(weights_nn, 3))

In [ ]:
# Visible on purpose: Exercise 4 and the explorers read these.
# Visible a propósito: el Ejercicio 4 y los exploradores los leen.

weights_nn, factors_nn = non_negative_parafac_hals(
    tl.tensor(X), rank=3, init="random", random_state=0,
    n_iter_max=300, tol=1e-8, normalize_factors=True)
A_nn, B_nn, C_nn = factors_nn
err_nn = float(np.linalg.norm(X - tl.cp_to_tensor((weights_nn, factors_nn)))
               / np.linalg.norm(X))

print("shapes / formas:", A_nn.shape, B_nn.shape, C_nn.shape)
print("weights / pesos:", np.round(weights_nn, 3))
print("relative error / error relativo:", round(err_nn, 4))
print("all factors non-negative? / ¿todos los factores no negativos?",
      bool(all((F >= 0).all() for F in factors_nn)))

### Interactive factor explorer / Explorador interactivo de factores

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

The three profiles of one component, side by side. The trial axis is sorted by the monkey's target, which the decomposition never saw — so any structure you see in the third panel is structure CP found on its own.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Los tres perfiles de una componente, uno al lado del otro. El eje de ensayos está ordenado por el objetivo del mono, que la descomposición nunca vio: cualquier estructura en el tercer panel es estructura que CP encontró sola.</div>

In [ ]:
#@title 🔍 Factor explorer / Explorador de factores { display-mode: 'form' }

import ipywidgets as widgets
from IPython.display import display

TARGETS = np.array([-90, 0, 90, 180])
trial_order = np.argsort([list(TARGETS).index(t) for t in angle], kind="stable")


def factor_draw(component, sort_trials):
    r = component
    figure, axes = plt.subplots(1, 3, figsize=(11.2, 2.9))
    axes[0].bar(np.arange(len(A_nn)), A_nn[:, r], color="#65a30d")
    axes[0].set_title("neurons / neuronas", fontsize=10)
    axes[1].plot(B_nn[:, r], lw=2, color="#65a30d")
    axes[1].axvline(int(np.argmax(B_nn[:, r])), color="#6b7280", ls="--", lw=1)
    axes[1].set_title(f"time / tiempo — peak at t={int(np.argmax(B_nn[:, r]))}",
                      fontsize=10)
    order = trial_order if sort_trials else np.arange(len(C_nn))
    axes[2].bar(np.arange(len(C_nn)), C_nn[order, r], color="#65a30d")
    if sort_trials:
        edges = np.cumsum([int((angle == t).sum()) for t in TARGETS])[:-1]
        for edge in edges:
            axes[2].axvline(edge - .5, color="#6b7280", lw=1)
    axes[2].set_title("trials / ensayos" + (" · grouped by target"
                                            if sort_trials else ""), fontsize=10)
    for ax in axes:
        ax.set_yticks([])
        ax.grid(alpha=.2)
    figure.suptitle(f"component {r} · weight {weights_nn[r]:.2f}", fontsize=11)
    figure.tight_layout()
    plt.show()


factor_pick = widgets.IntSlider(min=0, max=2, value=1,
                                description="component / componente:",
                                style={"description_width": "190px"},
                                continuous_update=False)
factor_sort = widgets.Checkbox(value=True, indent=False,
                               description="group trials by target / agrupa los "
                                           "ensayos por objetivo")

display(widgets.VBox([
    factor_pick, factor_sort,
    widgets.interactive_output(
        factor_draw, {"component": factor_pick, "sort_trials": factor_sort}),
]))

## Exercise 4 — the label the method never saw / Ejercicio 4 — la etiqueta que el método nunca vio

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

`angle` holds the target the monkey reached for on each of the 88 trials, one of `-90`, `0`, `90`, `180`. Nothing in the fit used it. The question is whether the trial factor found it anyway.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>angle</code> guarda el objetivo al que fue el mono en cada uno de los 88 ensayos: <code>-90</code>, <code>0</code>, <code>90</code> o <code>180</code>. Nada del ajuste lo usó. La pregunta es si el factor de ensayos lo encontró de todos modos.</div>

In [ ]:
# TODO 4 / TAREA 4
#
# 1. by_angle = a (3, 4) array: for each component r and each target t, the
#    mean of C_nn[:, r] over the trials whose angle is t.
#    Use TARGETS = np.array([-90, 0, 90, 180]) for the column order.
#
# 2. flat_component = the component whose four means are most nearly equal.
#    (Try the ratio of the row's spread to its mean.)
#
# 3. peak_times = for each component, the time step where B_nn peaks.
#
# 4. Write one sentence per component saying what it is: which neurons, when
#    in the trial, and on which trials. Then say which of the three could not
#    have been found without the behavioural labels — and whether it was.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }

TARGETS = np.array([-90, 0, 90, 180])

by_angle = np.array([[C_nn[angle == t, r].mean() for t in TARGETS]
                     for r in range(C_nn.shape[1])])

spread = np.ptp(by_angle, axis=1) / by_angle.mean(axis=1)
flat_component = int(np.argmin(spread))
peak_times = [int(np.argmax(B_nn[:, r])) for r in range(B_nn.shape[1])]

print("mean trial weight by target / peso medio por objetivo")
print("            ", "  ".join(f"{t:>6}" for t in TARGETS))
for r, row in enumerate(by_angle):
    print(f"component {r}:", "  ".join(f"{v:6.3f}" for v in row),
          f"   spread/mean {spread[r]:.2f}   peaks at t={peak_times[r]}")
print()
print("flattest component / componente más plana:", flat_component)

In [ ]:
# Visible on purpose: the wrap-up below reads these.
# Visible a propósito: el cierre de abajo los lee.

TARGETS = np.array([-90, 0, 90, 180])
by_angle = np.array([[C_nn[angle == t, r].mean() for t in TARGETS]
                     for r in range(C_nn.shape[1])])
spread = np.ptp(by_angle, axis=1) / by_angle.mean(axis=1)
flat_component = int(np.argmin(spread))
peak_times = [int(np.argmax(B_nn[:, r])) for r in range(B_nn.shape[1])]

print("flattest component / componente más plana:", flat_component,
      "  spread/mean:", round(float(spread[flat_component]), 3))
print("most tuned component / componente más selectiva:",
      int(np.argmax(spread)), "  spread/mean:", round(float(spread.max()), 3))
print("peak times / instantes de pico:", peak_times)

<details>
<summary><strong>What did Exercise 4 show? / ¿Qué mostró el Ejercicio 4?</strong></summary>

One of the three components has almost the same trial weight whatever the monkey was reaching for. Its time profile peaks early. That is a component describing something every trial does — the start of the movement, not its direction.

The other two are the opposite: each is strong for some targets and close to zero for others, and the two disagree about which. They peak later in the trial. Between them they carry the direction.

**Nothing in the fit used `angle`.** The decomposition was handed a `43 × 200 × 88` box of firing rates and a rank. It came back with a trial factor that sorts the trials by a variable measured in the monkey's behaviour, because that variable is genuinely part of what generated the numbers. This is the reason people fit CP to neural data, and it is worth being precise about what it is *not*: it is not a claim that component 1 "is" the 0° direction. It is a claim that a rank-3 summary of this tensor has to spend one of its three terms on something that varies with direction.

Careful about the third panel in the explorer. The bars are sorted by target *for display*; the fit saw them in recording order. If you untick the box the structure is still there, just scattered.

<br>

**ES.** Una de las tres componentes tiene casi el mismo peso de ensayo sea cual sea el objetivo, y su perfil temporal culmina pronto: describe algo que hacen todos los ensayos, el inicio del movimiento, no su dirección. Las otras dos son lo contrario: cada una es fuerte para unos objetivos y casi cero para otros, y discrepan entre sí; culminan más tarde. Entre las dos llevan la dirección. **Nada del ajuste usó `angle`**: a la descomposición se le dio una caja de `43 × 200 × 88` tasas de disparo y un rango, y devolvió un factor de ensayos que los ordena por una variable medida en la conducta, porque esa variable forma parte de lo que generó los números. Conviene ser preciso sobre lo que esto *no* es: no afirma que la componente 1 «sea» la dirección 0°, sino que un resumen de rango 3 de este tensor tiene que gastar uno de sus tres términos en algo que varía con la dirección. Ojo con el tercer panel: las barras se ordenan por objetivo *para verlas*; el ajuste las vio en orden de grabación.

</details>

## Exercise 5 — essentially unique, and what ALS returns / Ejercicio 5 — esencialmente única, y lo que devuelve ALS

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

Section 11 states Kruskal's result: under mild conditions a CP decomposition is **essentially unique** — determined except for the order of the `R` terms and a rescaling that cancels inside each one. That is a statement about the *decomposition*. ALS is an algorithm, and an algorithm that starts somewhere random has no obligation to honour it.

So: fit the same tensor three times from three different starting points and find out which of the two claims you are looking at.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La sección 11 enuncia el resultado de Kruskal: bajo condiciones suaves, una descomposición CP es <b>esencialmente única</b>, determinada salvo el orden de los <code>R</code> términos y un reescalado que se cancela dentro de cada uno. Eso habla de la <i>descomposición</i>. ALS es un algoritmo, y un algoritmo que empieza en un punto aleatorio no tiene obligación de respetarla. Ajusta el mismo tensor tres veces desde tres puntos y averigua cuál de las dos afirmaciones estás mirando.</div>

In [ ]:
# TODO 5 / TAREA 5
#
# 1. Fit non-negative CP at rank 3 three times, with random_state 0, 1 and 2.
#    Keep the weights, the factors and the relative error of each.
#
# 2. Are the three errors the same? Are the three weight vectors the same?
#    Are they the same *in the same order*?
#
# 3. Match the components of seed 1 against seed 0 by the correlation of their
#    time factors, and print the matching. Store it in matched.
#
# 4. Write one sentence distinguishing what Kruskal guarantees from what a run
#    of ALS guarantees.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }

runs = {}
for seed in (0, 1, 2):
    w, f = non_negative_parafac_hals(
        tl.tensor(X), rank=3, init="random", random_state=seed,
        n_iter_max=300, tol=1e-8, normalize_factors=True)
    runs[seed] = (w, f,
                  float(np.linalg.norm(X - tl.cp_to_tensor((w, f)))
                        / np.linalg.norm(X)))

for seed, (w, _, err) in runs.items():
    print(f"seed {seed}: error {err:.4f}   weights {np.round(np.sort(w)[::-1], 2)}"
          f"   as returned {np.round(w, 2)}")

B0 = runs[0][1][1]
B1 = runs[1][1][1]
corr = np.array([[abs(np.corrcoef(B0[:, i], B1[:, j])[0, 1]) for j in range(3)]
                 for i in range(3)])
matched = [int(np.argmax(corr[i])) for i in range(3)]

print()
print("time-factor |correlation| / |correlación| de los factores temporales")
print(np.round(corr, 3))
print("seed 0 component r matches seed 1 component / la componente r de la "
      "semilla 0 corresponde a:", matched)

<details>
<summary><strong>What did Exercise 5 show? / ¿Qué mostró el Ejercicio 5?</strong></summary>

Three different starting points, the same error to four decimal places, and the same three weights — **in a different order**. Match the components by their time profiles and every one lines up with a correlation of 1.

That is Kruskal's guarantee arriving intact, and it is worth noticing how narrow it is. It says nothing about the order the components come back in, so `factors[1]` from one run and `factors[1]` from another are not the same component and comparing them directly is a real mistake, easy to make in a loop. Always match components before comparing them, by correlation, by peak position, or by sorting on the weights.

It is also not a promise ALS makes. This tensor is well behaved; raise the rank past what the data supports and you can watch the same three seeds separate, which is what the seed slider on the ALS explorer is for. When runs *do* disagree, the decomposition is telling you the rank is wrong, not that the method failed.

<br>

**ES.** Tres puntos de partida distintos, el mismo error con cuatro decimales y los mismos tres pesos, **en otro orden**. Empareja las componentes por sus perfiles temporales y todas casan con correlación 1. Esa es la garantía de Kruskal llegando intacta, y conviene ver lo estrecha que es: no dice nada del orden en que vuelven las componentes, así que `factors[1]` de una ejecución y `factors[1]` de otra no son la misma componente, y compararlas directamente es un error real y fácil de cometer en un bucle. Empareja siempre antes de comparar: por correlación, por posición del pico o ordenando por los pesos. Tampoco es una promesa de ALS: este tensor se porta bien, pero sube el rango por encima de lo que los datos sostienen y verás separarse a las mismas tres semillas. Cuando las ejecuciones discrepan, la descomposición te está diciendo que el rango está mal, no que el método falló.

</details>

## What just happened / Qué acaba de pasar

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

You wrote CP down as a sum of rank-1 terms, and then established that the only readable things in one term are the **normalised profiles** and a single number holding the size. Everything else is an artefact of how you happened to split the scale.

You fitted it yourself. The full problem is not convex; fixing all but one factor makes it an ordinary least-squares problem, which is section 07's pseudoinverse and nothing more. Alternating those three solves is the whole algorithm, the error can never rise, and the price is that where you start decides where you stop.

You added non-negativity and watched the subproblem turn from a closed form into NNLS — still convex, still one factor at a time — and paid a little squared error for factors that can be described in words.

And you read the result: a rank-3 summary of 43 neurons, 200 time steps and 88 trials, in which one component is what every trial does and two carry the direction the monkey reached for. The decomposition was never told which direction that was.

**The sentence to remember:** *a CP component is one profile per axis, and its peaks are where that axis's story happens.*

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Escribiste CP como suma de términos de rango 1 y estableciste que lo único legible de un término son los <b>perfiles normalizados</b> y un solo número con el tamaño; el resto es un artefacto de cómo repartiste la escala.
<br><br>
Lo ajustaste tú. El problema completo no es convexo; fijar todos los factores menos uno lo convierte en mínimos cuadrados corrientes, que es la pseudoinversa de la sección 07 y nada más. Alternar esas tres soluciones es todo el algoritmo, el error no puede subir, y el precio es que dónde empiezas decide dónde paras.
<br><br>
Añadiste no negatividad y viste el subproblema pasar de forma cerrada a NNLS — convexo todavía, un factor cada vez — y pagaste algo de error cuadrático por factores que se pueden describir con palabras.
<br><br>
Y leíste el resultado: un resumen de rango 3 de 43 neuronas, 200 instantes y 88 ensayos, en el que una componente es lo que hacen todos los ensayos y dos llevan la dirección a la que fue el mono. A la descomposición nunca se le dijo cuál era esa dirección.
<br><br>
<b>La frase que recordar:</b> <i>una componente CP es un perfil por eje, y sus picos son donde ocurre la historia de ese eje.</i></div>

Deep dive 15 asks the question this notebook never did: squared error assumed something about the data. What if it was wrong?

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#65a30d,rgba(101,163,13,0))"></div>

## Done with this deep dive / Fin de este estudio a fondo

Next deep dive / Siguiente estudio a fondo: **15 · Generalized CP for counts and binary data / CP generalizado para conteos y datos binarios** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/15-generalized-cp.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)